# Step 3 — VLM Caption Overlay (Ollama)

This notebook runs the Ollama VLM to generate short captions and overlays them on annotated videos produced in Step 1 (YOLO) or Step 2 (CLIP).

Prerequisites:

- `ollama` installed and running (`ollama serve`),
- Model pulled: `ollama pull qwen3.5:4b` or `ollama pull qwen3-vl:4b`,
- Python client installed in the notebook environment: `pip install ollama`.

Inputs:

- Default input root: `runs/detect/` (looks for `predict*` folders inside).
- You can override with `PREDICT_DIR` (full folder path) or `INPUT_ROOT` (root containing `predict*`).
- You can override the exact video with `INPUT_VIDEO` (full file path).

Captioning:

- Default mode is **adaptive** (caption only when the scene changes),
- A minimum gap of **5 seconds** is enforced so captions are readable.

Place this notebook next to the YOLO project and run after Step 1 and Step 2 produce `runs/detect/predict*` outputs.

In [1]:
!pip install -q ollama

In [4]:
# VLM caption overlay (Ollama) - Step 3
import io
import os
import glob
import textwrap
import cv2
from PIL import Image

try:
    from ollama import chat
except ImportError as exc:
    raise ImportError("Missing ollama Python client. Install it in your environment: pip install ollama") from exc

# Configuration (fixed model for this project)
VLM_MODEL = "qwen3-vl:4b"
VLM_PROMPT = "Describe the driving scene in one short sentence."
PREDICT_DIR = os.getenv("PREDICT_DIR", "")
INPUT_ROOT = os.getenv("INPUT_ROOT", "runs/detect")
INPUT_VIDEO = os.getenv("INPUT_VIDEO", "")
VLM_MAX_ERRORS = int(os.getenv("VLM_MAX_ERRORS", "5"))

# If you move to a cloud VLM later, do this:
# 1) Set a host or API base URL in your terminal (example):
#    export OLLAMA_HOST="http://<host>:<port>"
# 2) If the provider needs an API key, set it in the terminal:
#    export VLM_API_KEY="<your_key>"
# 3) Update the client call in get_caption() to use that provider's SDK.

# Captioning controls
CAPTION_MODE = os.getenv("CAPTION_MODE", "adaptive")  # adaptive = scene change, fixed = every N seconds
CAPTION_EVERY_SEC = float(os.getenv("CAPTION_EVERY_SEC", "8.0"))  # used only in fixed mode
CAPTION_MIN_SEC = float(os.getenv("CAPTION_MIN_SEC", "5.0"))  # minimum gap between captions
CAPTION_DIFF_THRESHOLD = float(os.getenv("CAPTION_DIFF_THRESHOLD", "12.0"))  # higher = fewer updates

def frame_to_bytes(frame_bgr):
    # Convert OpenCV BGR frame to JPEG bytes for Ollama
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return buf.getvalue()

def get_caption(frame_bgr):
    # Send one frame to the VLM and get a single-sentence caption
    # If you switch providers, replace chat(...) below with that SDK call.
    img_bytes = frame_to_bytes(frame_bgr)
    response = chat(
        model=VLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": VLM_PROMPT,
                "images": [img_bytes],
            }
        ],
        stream=False,
    )
    return response.message.content.strip().replace("\n", " ")

def overlay_caption(frame_bgr, caption):
    # Draw the caption on a black banner for readability
    if not caption:
        return frame_bgr
    lines = textwrap.wrap(caption, width=40)
    font_scale = 3.0
    line_height = 120
    pad = 50
    thickness = 10
    box_height = pad * 2 + line_height * len(lines)
    cv2.rectangle(frame_bgr, (0, 0), (frame_bgr.shape[1], box_height), (0, 0, 0), -1)
    y = pad + line_height
    for line in lines:
        cv2.putText(
            frame_bgr,
            line,
            (10, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )
        y += line_height
    return frame_bgr

def resolve_input_video():
    # Find the newest annotated video under runs/detect/predict* if none is specified
    if INPUT_VIDEO:
        if not os.path.isfile(INPUT_VIDEO):
            raise FileNotFoundError(f"INPUT_VIDEO not found: {INPUT_VIDEO}")
        return INPUT_VIDEO
    if PREDICT_DIR:
        if not os.path.isdir(PREDICT_DIR):
            raise FileNotFoundError(f"PREDICT_DIR not found: {PREDICT_DIR}")
        predict_dir = PREDICT_DIR
    else:
        predict_dirs = glob.glob(os.path.join(INPUT_ROOT, "predict*"))
        if not predict_dirs:
            raise FileNotFoundError(
                f"No predict* folders found under {INPUT_ROOT}. Set INPUT_ROOT or PREDICT_DIR."
            )
        predict_dir = max(predict_dirs, key=os.path.getmtime)
    video_candidates = glob.glob(os.path.join(predict_dir, "*.avi")) + glob.glob(
        os.path.join(predict_dir, "*.mp4")
    )
    if not video_candidates:
        raise FileNotFoundError(f"No video files found in {predict_dir}")
    return max(video_candidates, key=os.path.getmtime)

def should_caption_fixed(frame_idx, interval_frames):
    return frame_idx % interval_frames == 0

def should_caption_adaptive(frame_bgr, last_frame_bgr, elapsed_sec):
    # Update only if enough time passed AND the scene changed enough
    if last_frame_bgr is None:
        return True
    if elapsed_sec < CAPTION_MIN_SEC:
        return False
    gray_now = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    gray_last = cv2.cvtColor(last_frame_bgr, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(gray_now, gray_last)
    mean_diff = float(diff.mean())
    return mean_diff >= CAPTION_DIFF_THRESHOLD

input_video = resolve_input_video()
base_name = os.path.splitext(os.path.basename(input_video))[0]
out_dir = os.path.join(os.path.dirname(input_video), "vlm_overlay")
os.makedirs(out_dir, exist_ok=True)
output_video = os.path.join(out_dir, f"{base_name}_vlm.mp4")

cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise RuntimeError(f"Failed to open video: {input_video}")
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
interval_frames = max(1, int(round(fps * CAPTION_EVERY_SEC)))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

frame_idx = 0
current_caption = ""
last_caption_frame = None
last_caption_idx = -999999
error_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    elapsed_sec = (frame_idx - last_caption_idx) / fps
    if CAPTION_MODE == "adaptive":
        do_caption = should_caption_adaptive(frame, last_caption_frame, elapsed_sec)
    else:
        do_caption = should_caption_fixed(frame_idx, interval_frames)
    if do_caption:
        try:
            current_caption = get_caption(frame)
            last_caption_frame = frame.copy()
            last_caption_idx = frame_idx
        except Exception as exc:
            error_count += 1
            print(f"VLM error at frame {frame_idx}: {exc} (errors={error_count})")
            if VLM_MAX_ERRORS > 0 and error_count >= VLM_MAX_ERRORS:
                raise RuntimeError(f"VLM failed {error_count} times; aborting.")
    frame = overlay_caption(frame, current_caption)
    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()
print(f"Input:  {input_video}")
print(f"Output: {output_video}")

Input:  runs/detect/predict-2/dashcam.avi
Output: runs/detect/predict-2/vlm_overlay/dashcam_vlm.mp4


Next steps:

- Optionally add `try/except` around `chat()` to handle `ollama.ResponseError` and connection errors.
- If you want host control, set `OLLAMA_HOST` or provide a `Client(host=...)` from the `ollama` library.